In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt

In [2]:
birds=pd.read_csv("birds_noisy_dataset.csv")
birds_conservation=pd.read_csv("bird_conservation.csv")
observers=pd.read_csv("observers.csv")

In [3]:
birds.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 90 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bird_id           80000 non-null  object 
 1   species           74319 non-null  object 
 2   family            74239 non-null  object 
 3   continent         74429 non-null  object 
 4   country           74215 non-null  object 
 5   habitat           74241 non-null  object 
 6   diet              74283 non-null  object 
 7   wing_span_cm      72016 non-null  float64
 8   weight_g          71926 non-null  float64
 9   beak_length_mm    71906 non-null  float64
 10  tail_length_cm    71970 non-null  float64
 11  age_years         72088 non-null  float64
 12  gender            74243 non-null  object 
 13  migratory         74317 non-null  object 
 14  observation_year  75909 non-null  float64
 15  observer_id       39889 non-null  object 
 16  location_lat      71974 non-null  float6

In [4]:
# standardizing the column names
birds.columns=(birds.columns
               .str.strip()
               .str.lower()
               .str.replace(" ","_")
               )

In [5]:
# remove the duplicate rows
birds.drop_duplicates(inplace=True)

In [6]:
#replace the dirty data with np.nan
birds.replace(["unknown","Unknown","?","NA","N/A","none","","UNKNOWN","NONE","Null", "None"], np.nan, inplace=True)

In [7]:
# threshold for dropping the columns if contain more missing values
threshold=len(birds)*0.6
birds = birds.dropna(thresh=threshold, axis=1)

In [19]:
#setting the datatypes
birds_col=[
           "wing_span_cm", 
           "weight_g", 
           "beak_length_mm", 
           "tail_length_cm", 
           "age_years",  
           "temperature_c"]
for col in birds_col:
    birds[col] = pd.to_numeric(birds[col], errors="coerce")

birds_cateog=["species","family","continent", "country","habitat", "diet",
              "gender","migratory","weather"]
features=[col for col in birds.columns if "feature" in col]
for col in birds_cateog:
    birds[col] = birds[col].str.lower().astype("category")

for col in features:
    birds[col] = birds[col].astype("category")

birds["observation_year"] = pd.to_datetime(birds["observation_year"], errors="coerce")

In [9]:
# replacing the nan values with mean
mean_wing_span=birds["wing_span_cm"].mean() #7984 rows × 90 columns
birds["wing_span_cm"].fillna(mean_wing_span, inplace=True)

mean_weight_g=birds["weight_g"].mean() 
birds["weight_g"].fillna(mean_weight_g, inplace=True)

mean_beak_length_mm=birds["beak_length_mm"].mean() 
birds["beak_length_mm"].fillna(mean_beak_length_mm, inplace=True)

mean_tail_length_cm=birds["tail_length_cm"].mean() 
birds["tail_length_cm"].fillna(mean_tail_length_cm, inplace=True)

mean_temperature_c=birds["temperature_c"].mean() 
birds["temperature_c"].fillna(mean_temperature_c, inplace=True)

mean_age_years=birds["age_years"].mean() 
birds["age_years"].fillna(mean_age_years, inplace=True)

# replaced way
cols = [
    "wing_span_cm",
    "weight_g",
    "beak_length_mm",
    "tail_length_cm",
    "temperature_c",
    "age_years"
]
birds[cols] = birds[cols].fillna(birds[cols].mean())

C:\Users\PMLS\AppData\Local\Temp\ipykernel_14864\2854830083.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  birds["wing_span_cm"].fillna(mean_wing_span, inplace=True)
C:\Users\PMLS\AppData\Local\Temp\ipykernel_14864\2854830083.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


In [10]:
num_cols = birds.select_dtypes(include=["float64"]).columns

for col in num_cols:
    birds[col] = pd.to_numeric(birds[col], downcast="float")

In [11]:
birds.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 89 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   bird_id           80000 non-null  object        
 1   species           65923 non-null  category      
 2   family            60967 non-null  category      
 3   continent         62975 non-null  category      
 4   country           64015 non-null  category      
 5   habitat           62786 non-null  category      
 6   diet              55075 non-null  category      
 7   wing_span_cm      80000 non-null  float32       
 8   weight_g          80000 non-null  float32       
 9   beak_length_mm    80000 non-null  float32       
 10  tail_length_cm    80000 non-null  float32       
 11  age_years         80000 non-null  float32       
 12  gender            49361 non-null  category      
 13  migratory         49051 non-null  category      
 14  observation_year  7590

In [12]:
missing_report = birds.isna().sum().sort_values(ascending=False)
print(missing_report.head(20))

migratory     30949
gender        30639
diet          24925
family        19033
weather       18925
habitat       17214
continent     17025
country       15985
species       14077
feature_49    10493
feature_16    10489
feature_25    10422
feature_29    10419
feature_61    10410
feature_2     10405
feature_28    10395
feature_51    10389
feature_68    10387
feature_9     10372
feature_56    10367
dtype: int64


Some advanced things

query()

In [13]:
birds[birds["weight_g"] > 1000]
birds.query("weight_g > 1000")# this is better
birds.query("weight_g > 1000 and wing_span_cm > 80")

,bird_id,species,family,continent,country,habitat,diet,wing_span_cm,weight_g,beak_length_mm,...,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70
630,B000630,Pigeon,NaN,Australia,Canada,Desert,Carnivore,98.832458,1063.727051,36.616734,...,692,872,NaN,342.17,453,NaN,NaN,NaN,NaN,NaN
1473,B001473,NaN,Psittacidae,North America,NaN,Grassland,NaN,101.188736,1213.771240,34.885185,...,310,333.29,349.05,130.85,664,213.89,371.29,8.37,425,NaN
3416,B003416,Parrot,Accipitridae,North America,UK,Desert,Herbivore,93.729904,1040.805542,55.761726,...,NaN,251.46,291.5,205.27,NaN,NaN,150.91,NaN,828,218.2
5051,B005051,Pigeon,Corvidae,NaN,NaN,Wetland,Omnivore,109.093475,1064.682129,51.247379,...,359,907,204.94,228.58,NaN,240,342.49,864,missing,407
5864,B005864,Peacock,NaN,Europe,UK,Forest,Herbivore,90.307243,1021.551025,62.827255,...,405.84,87,317.43,266.89,584,436.04,840,452,360,358.06
6235,B006235,Falcon,Strigidae,North America,USA,Desert,Omnivore,88.426994,1015.768188,50.266499,...,74.59,104.65,18.55,241,NaN,332,41.8,75.13,374.16,393
7116,B007116,missing,Accipitridae,South America,Brazil,Wetland,Carnivore,103.076050,1018.090576,36.740158,...,342.43,missing,48,NaN,NaN,434,621,NaN,256.47,NaN
11257,B011257,Owl,Psittacidae,NaN,Australia,Desert,Omnivore,83.850090,1012.940002,51.460590,...,362.37,90.74,308,475.45,274,NaN,73,351.91,358,544
12107,B012107,NaN,Strigidae,South America,UK,NaN,Omnivore,96.872993,1039.387451,59.176422,...,NaN,missing,NaN,272.27,104.54,478.86,131.2,480.21,977,577
12394,B012394,Parrot,Strigidae,NaN,Brazil,Wetland,NaN,87.639610,1006.912720,65.255058,...,15.03,651,965,136.18,257.25,329,64.9,207.29,99,138.15


assign() (Create New Columns Cleanly)

In [14]:
birds = birds.assign(
    wing_weight_ratio = birds["wing_span_cm"] / birds["weight_g"]
)

value_counts(normalize=True)->percentage distribution

In [15]:
birds["species"].value_counts(normalize=True)

species
Peacock    0.111782
Eagle      0.110872
Falcon     0.110174
Duck       0.110144
Pigeon     0.109704
Owl        0.108642
Parrot     0.107747
Crow       0.107595
Sparrow    0.106124
missing    0.017217
Name: proportion, dtype: float64

nlargest/nsmallest->instead of sorting entire data set use nlargest/smallest to sort just one column

In [16]:
# 1.	Load the dataset using pandas.
#already done

# 2.	Display the first 10 rows and last 10 rows.
first_10=birds.head(10)
last_10=birds.tail(10)

# 3.	Find the shape of the dataset.
rows_columns=birds.shape

# 4.	List all column names.
column_names=birds.columns

# 5.	Check the datatype of each column.
datatypes=birds.dtypes

# 6.	Count total missing values in each column.
missing_values=birds.isna().sum()

# 7.	Identify columns containing "unknown", "missing", "NULL" or empty strings.
values_to_check = ["unknown", "missing", "NULL", ""]
columns_with_values = birds.isin(values_to_check).any()


In [17]:
# 8.	Replace "unknown", "missing", "NULL", "nan" with proper NaN.
birds.replace(["unknown", "missing", "NULL", "nan"], np.nan, inplace=True)
# 9.	Convert numeric columns like wing_span_cm, weight_g, beak_length_mm to numeric datatype.
birds_col=[
           "wing_span_cm", 
           "weight_g", 
           "beak_length_mm", 
           "tail_length_cm", 
           "age_years",  
           "temperature_c"]
for col in birds_col:
    birds[col] = pd.to_numeric(birds[col], errors="coerce")

# 10.	Drop rows where bird_id is missing.
birds.dropna(subset=["birds_id"], axis=1)

# 11.	Fill missing temperature_c using column mean.
birds["temperature_c"].mean()

# 12.	Replace missing country with "Unknown Country".
birds["country"].fillna("Unknown Country", inplace=True)

# 13.	Remove duplicate rows.
birds.drop_duplicates(inplace=True)

# 14.	Standardize text columns to lowercase.
birds.columns=(birds.columns
               .str.strip()
               .str.lower()
               .str.replace(" ","_")
               )

# 15.	Detect outliers in weight_g.
outliers = birds[birds["weight_g"] < 0]
print(outliers)

C:\Users\PMLS\AppData\Local\Temp\ipykernel_14864\2008088307.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  birds.replace(["unknown", "missing", "NULL", "nan"], np.nan, inplace=True)


KeyError: ['birds_id']

In [ ]:
birds.columns

Index(['bird_id', 'species', 'family', 'continent', 'country', 'habitat',
       'diet', 'wing_span_cm', 'weight_g', 'beak_length_mm', 'tail_length_cm',
       'age_years', 'gender', 'migratory', 'observation_year', 'location_lat',
       'location_long', 'weather', 'temperature_c', 'feature_1', 'feature_2',
       'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7',
       'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12',
       'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17',
       'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22',
       'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27',
       'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32',
       'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37',
       'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42',
       'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47',
    

In [ ]:
# Basic GroupBy
# 16.	Count number of birds per species.
birds.groupby("species").size()
# 17.	Find average wing span per species.
birds.groupby("species")["wing_span_cm"].mean()
# 18.	Find average weight per habitat.
birds.groupby("habitat")["weight_g"].mean()
# 19.	Count birds observed per country.
birds.groupby("country").size()
# 20.	Count birds observed per year.
birds.groupby("observation_year").size()


C:\Users\PMLS\AppData\Local\Temp\ipykernel_14096\4140909275.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birds.groupby("species").size()
C:\Users\PMLS\AppData\Local\Temp\ipykernel_14096\4140909275.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birds.groupby("species")["wing_span_cm"].mean()
C:\Users\PMLS\AppData\Local\Temp\ipykernel_14096\4140909275.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birds.gr

observation_year
1970-01-01 00:00:00.000002000    2951
1970-01-01 00:00:00.000002001    2890
1970-01-01 00:00:00.000002002    2936
1970-01-01 00:00:00.000002003    2882
1970-01-01 00:00:00.000002004    2808
1970-01-01 00:00:00.000002005    2956
1970-01-01 00:00:00.000002006    2916
1970-01-01 00:00:00.000002007    2848
1970-01-01 00:00:00.000002008    2889
1970-01-01 00:00:00.000002009    2915
1970-01-01 00:00:00.000002010    2935
1970-01-01 00:00:00.000002011    2969
1970-01-01 00:00:00.000002012    2871
1970-01-01 00:00:00.000002013    2908
1970-01-01 00:00:00.000002014    2969
1970-01-01 00:00:00.000002015    3011
1970-01-01 00:00:00.000002016    2944
1970-01-01 00:00:00.000002017    2960
1970-01-01 00:00:00.000002018    3010
1970-01-01 00:00:00.000002019    2908
1970-01-01 00:00:00.000002020    2870
1970-01-01 00:00:00.000002021    2856
1970-01-01 00:00:00.000002022    2863
1970-01-01 00:00:00.000002023    2981
1970-01-01 00:00:00.000002024    2939
1970-01-01 00:00:00.000002025    

In [ ]:
#correcting the weight column
birds[birds["weight_g"]<0]
mean_weight_g=birds[birds["weight_g"]>0]["weight_g"].mean()
birds.loc[birds["weight_g"]<0, "weight_g"]=mean_weight_g

In [ ]:
# beak_length_mm drop them because only 4 rows
birds.drop(birds[birds["beak_length_mm"] < 0].index, inplace=True)

In [ ]:
#relace the mnegative values with mean
birds_tail_length_mean=birds[birds["tail_length_cm"]>0]["tail_length_cm"].mean()
birds.loc[birds["tail_length_cm"]<0, "tail_length_cm"]=birds_tail_length_mean

In [ ]:
# Multi-Level GroupBy
# 21.	Find average weight_g grouped by species and habitat.
birds.groupby(["species","habitat"])["weight_g"].mean()
# 22.	Find maximum wing_span_cm grouped by continent and species.
mean_wing_span=birds[birds["wing_span_cm"]>0]["wing_span_cm"].mean()
birds.loc[birds["wing_span_cm"]<0, "wing_span_cm"]=mean_wing_span
birds.groupby(["continent","species"])["wing_span_cm"].max()

# 23.	Count birds grouped by country and diet.
birds.groupby(["country","diet"]).size()

# 24.	Find mean temperature_c grouped by weather and habitat.
birds.groupby(["weather","habitat"])["temperature_c"].mean()

# 25.	Find total birds per continent and migratory status.
birds.groupby(["continent","migratory"]).size()

C:\Users\PMLS\AppData\Local\Temp\ipykernel_21956\1508149832.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birds.groupby(["species","habitat"])["weight_g"].mean()
C:\Users\PMLS\AppData\Local\Temp\ipykernel_21956\1508149832.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birds.groupby(["continent","species"])["wing_span_cm"].max()
C:\Users\PMLS\AppData\Local\Temp\ipykernel_21956\1508149832.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future defaul

continent      migratory
Africa         No           3141
               Yes          3182
Asia           No           3101
               Yes          3125
Australia      No           3051
               Yes          3069
Europe         No           3041
               Yes          3037
North America  No           3054
               Yes          3063
South America  No           3113
               Yes          3107
dtype: int64

In [ ]:

# Advanced GroupBy
# 26.	Find top 5 species with highest average wing span.
top5_wing_species = birds.groupby("species")["wing_span_cm"].mean().nlargest(5)
print(top5_wing_species)
# 27.	Find species with highest average weight in each continent.
species_max_weight_continent = birds.groupby("continent").apply(
    lambda x: x.groupby("species")["weight_g"].mean().idxmax()
)
print(species_max_weight_continent)
# 28.	Find habitat where birds live longest on average.
habitat_longest_lived = birds.groupby("habitat")["age_years"].mean().idxmax()
print("Habitat where birds live longest on average:", habitat_longest_lived)
# 29.	Find number of male vs female birds per species.
gender_count_species = birds.groupby(["species","gender"]).size()
print(gender_count_species)
# 30.	Find species distribution per year.
species_distribution_year = birds.groupby(["observation_year","species"]).size()
print(species_distribution_year)

species
Falcon     60.412338
Crow       60.304989
Parrot     60.269287
Sparrow    60.259197
Eagle      60.194481
Name: wing_span_cm, dtype: float32
continent
Africa            Eagle
Asia             Falcon
Australia           Owl
Europe             Duck
North America       Owl
South America     Eagle
dtype: object
Habitat where birds live longest on average: Mountain
species  gender
Crow     Female    2178
         Male      2121
Duck     Female    2119
         Male      2198
Eagle    Female    2204
         Male      2197
Falcon   Female    2208
         Male      2105
Owl      Female    2172
         Male      2179
Parrot   Female    2116
         Male      2199
Peacock  Female    2291
         Male      2179
Pigeon   Female    2192
         Male      2155
Sparrow  Female    2076
         Male      2097
dtype: int64
observation_year               species
1970-01-01 00:00:00.000002000  Crow       265
                               Duck       291
                               Eagle  

C:\Users\PMLS\AppData\Local\Temp\ipykernel_21956\2276223107.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top5_wing_species = birds.groupby("species")["wing_span_cm"].mean().nlargest(5)
C:\Users\PMLS\AppData\Local\Temp\ipykernel_21956\2276223107.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  species_max_weight_continent = birds.groupby("continent").apply(
C:\Users\PMLS\AppData\Local\Temp\ipykernel_21956\2276223107.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True

In [ ]:
birds.columns

Index(['bird_id', 'species', 'family', 'continent', 'country', 'habitat',
       'diet', 'wing_span_cm', 'weight_g', 'beak_length_mm', 'tail_length_cm',
       'age_years', 'gender', 'migratory', 'observation_year', 'location_lat',
       'location_long', 'weather', 'temperature_c', 'feature_1', 'feature_2',
       'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7',
       'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12',
       'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17',
       'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22',
       'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27',
       'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32',
       'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37',
       'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42',
       'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47',
    

In [ ]:
birds_conservation.columns

Index(['species', 'conservation_status', 'population_estimate', 'threat_level',
       'protected_regions', 'last_assessment_year'],
      dtype='object')

In [ ]:
# 31.	Merge bird dataset with conservation dataset. 
bdxconservation=pd.concat((birds,birds_conservation), axis=0) #adding the columns->horizontal stacking
# 33.	Perform left join between bird data and conservation data. (complete left, common rows of right and all columns of both)
birds.merge(birds_conservation, how="left", on="species")
# 34.	Perform right join. (complete right, common rows of left and all columns of both)
birds.merge(birds_conservation, how="right", on="species") 
# 35.	Perform inner join.(all columns of both, common rows of both )
birds.merge(birds_conservation, how="inner", on="species") 
# 36.	Perform outer join.(all rows, all columns)
birds.merge(birds_conservation, how="outer", on="species") 
# 37.	Identify birds with missing conservation status after merge.
bdxconservation[bdxconservation["conservation_status"]=="Unknown"]

In [ ]:
bdxconservation=pd.concat((birds,birds_conservation), axis=0)
bdxconservation.shape

(80196, 95)

In [18]:
# Concat Practice
# 21.	Split birds dataset into:
# Asia birds
# Europe birds
# Africa birds
# Then:
# 22.	Concatenate vertically.
# 23.	Concatenate horizontally.
# 24.	Reset index after concat.
asia_birds = birds[birds["continent"] == "Asia"]
europe_birds = birds[birds["continent"] == "Europe"]
africa_birds = birds[birds["continent"] == "Africa"]
vertical_concat = pd.concat([asia_birds, europe_birds, africa_birds], axis=0)
horizontal_concat = pd.concat([asia_birds, europe_birds, africa_birds], axis=1)
vertical_concat = vertical_concat.reset_index(drop=True)